[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch05-nn-computation/01-lan-truyen-nguoc.ipynb)

# Lan truyền ngược bằng tay, rồi để autograd kiểm tra

Chương 5 lần theo gradient qua một mạng tí hon: hai đầu vào, hai nơ-ron ẩn dùng ReLU, một đầu ra.
Sổ tay này làm lại đúng phép tính đó bằng NumPy, từng bước của quy tắc dây chuyền, rồi đối
chiếu với ba nguồn độc lập: con số trong sách, autograd của PyTorch, và sai phân hữu hạn.

Phần cuối dùng cùng cách đếm cho mạng nhận dạng chữ số 784-128-64-10 của chương: bao nhiêu
tham số, bao nhiêu phép nhân-cộng, và bộ nhớ huấn luyện lớn hơn bộ nhớ suy luận bao nhiêu.

**Bạn sẽ làm:**
1. lượt truyền xuôi của mạng tí hon;
2. lượt truyền ngược, năm bước;
3. so với autograd và với sai phân hữu hạn;
4. đếm tham số, MAC và bộ nhớ của mạng 784-128-64-10.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    return dung


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Lượt truyền xuôi

Mạng và các giá trị khởi đầu lấy đúng từ Napkin Math 1.3 của chương: đầu vào
$\mathbf{x} = [1.0,\ 0.5]$, nhãn $y = 1.0$, hàm mất mát $\mathcal{L} = \tfrac12(\hat y - y)^2$.
Tầng ẩn nhân $\mathbf{x}$ với $\mathbf{W}^{(1)}$ rồi đi qua ReLU, tầng ra nhân với
$\mathbf{W}^{(2)}$ và không có hàm kích hoạt.

In [ ]:
x = np.array([1.0, 0.5])
y = 1.0
W1 = np.array([[0.5, -0.3],
               [0.8, 0.2]])
b1 = np.zeros(2)
W2 = np.array([0.6, -0.4])
b2 = 0.0

z1 = x @ W1 + b1            # tiền kích hoạt của tầng ẩn
a1 = np.maximum(0.0, z1)    # ReLU
y_hat = a1 @ W2 + b2        # đầu ra
L = 0.5 * (y_hat - y) ** 2  # sai số bình phương, nhân một nửa

print("z1 =", z1, " a1 =", a1)
theo_sach("z1, nơ-ron ẩn thứ nhất", z1[0], sach=0.9, nguon="0,9 / −0,2")
theo_sach("z1, nơ-ron ẩn thứ hai", z1[1], sach=-0.2, nguon="0,9 / −0,2")
theo_sach("đầu ra ŷ", y_hat, sach=0.54, nguon="0,54")
theo_sach("mất mát L", L, sach=0.1058, nguon="0,1058")

### Dự đoán trước khi chạy tiếp

Nơ-ron ẩn thứ hai có $z = -0.2$, nên ReLU trả về 0. Trọng số **đi vào** nơ-ron đó sẽ nhận
gradient bằng bao nhiêu? Hãy đoán trước, rồi xem bước 4 và 5 bên dưới.

## 2. Lượt truyền ngược, từng bước

Quy tắc dây chuyền đi từ mất mát ngược về đầu vào. Mỗi bước dùng lại một giá trị mà lượt
truyền xuôi đã lưu: đó là lý do huấn luyện phải giữ giá trị kích hoạt trong bộ nhớ.

In [ ]:
# Bước 1: gradient của mất mát theo đầu ra.
dL_dyhat = y_hat - y
# Bước 2: tầng ra không có hàm kích hoạt, nên dL/dz2 = dL/dŷ; gradient trọng số là a1ᵀ · dL/dz2.
dL_dW2 = a1 * dL_dyhat
dL_db2 = dL_dyhat
# Bước 3: tín hiệu lỗi gửi ngược về tầng ẩn.
dL_da1 = dL_dyhat * W2
# Bước 4: qua ReLU. Đạo hàm là 1 ở nơi z > 0 và 0 ở nơi còn lại.
dL_dz1 = dL_da1 * (z1 > 0)
# Bước 5: gradient của trọng số tầng ẩn là xᵀ · dL/dz1 (một tích ngoài).
dL_dW1 = np.outer(x, dL_dz1)
dL_db1 = dL_dz1

theo_sach("dL/dŷ", dL_dyhat, sach=-0.46, nguon="−0,46")
theo_sach("dL/dW2, phần tử thứ nhất", dL_dW2[0], sach=-0.414, nguon="−0,414")
theo_sach("dL/da1, nơ-ron thứ nhất", dL_da1[0], sach=-0.276, nguon="−0,276 / 0,184")
theo_sach("dL/da1, nơ-ron thứ hai", dL_da1[1], sach=0.184, nguon="−0,276 / 0,184")
print("dL/dz1 =", dL_dz1)
print("dL/dW1 =\n", dL_dW1)

Đúng như dự đoán ở trên: nơ-ron thứ hai bị ReLU chặn ở lượt truyền xuôi, nên ở lượt truyền
ngược nó cũng chặn luôn tín hiệu lỗi. Cả cột thứ hai của $\partial\mathcal{L}/\partial\mathbf{W}^{(1)}$
bằng 0, và các trọng số đó đứng yên ở bước cập nhật này.

Một bước cập nhật với tốc độ học $\eta$ đẩy mỗi trọng số ngược chiều gradient của nó:

In [ ]:
eta = sach(0.1, "η = 0,1")
W1_moi = W1 - eta * dL_dW1
theo_sach("W1[1,1] sau một bước", W1_moi[0, 0], sach=0.5276, nguon="0,5 → 0,5276")
print("W1 sau một bước =\n", W1_moi)

## 3. Ba cách tính, một đáp số

Gradient tính bằng tay chỉ đáng tin khi một cách độc lập cho cùng kết quả. Có hai cách như thế:

* **autograd** của PyTorch ghi lại đồ thị tính toán ở lượt truyền xuôi, rồi tự áp quy tắc
  dây chuyền khi gọi `backward()`;
* **sai phân hữu hạn** không dùng đạo hàm nào cả: nhích một trọng số đi một lượng rất nhỏ
  $\varepsilon$ và xem mất mát đổi bao nhiêu, $\frac{\mathcal{L}(w+\varepsilon)-\mathcal{L}(w-\varepsilon)}{2\varepsilon}$.

In [ ]:
tW1 = torch.tensor(W1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tx = torch.tensor(x)
t_loss = 0.5 * (torch.relu(tx @ tW1) @ tW2 - y) ** 2
t_loss.backward()

print("autograd dL/dW1 =\n", tW1.grad.numpy())
lech_autograd = max(np.abs(tW1.grad.numpy() - dL_dW1).max(), np.abs(tW2.grad.numpy() - dL_dW2).max())
print(f"lệch lớn nhất so với tính tay: {lech_autograd:.1e}")
assert lech_autograd < 1e-12

In [ ]:
def mat_mat(W1_, W2_):
    return 0.5 * (np.maximum(0.0, x @ W1_) @ W2_ - y) ** 2


eps = 1e-6
so_hoc = np.zeros_like(W1)
for i in range(2):
    for j in range(2):
        cong, tru = W1.copy(), W1.copy()
        cong[i, j] += eps
        tru[i, j] -= eps
        so_hoc[i, j] = (mat_mat(cong, W2) - mat_mat(tru, W2)) / (2 * eps)

print("sai phân hữu hạn dL/dW1 =\n", so_hoc)
lech_sai_phan = np.abs(so_hoc - dL_dW1).max()
print(f"lệch lớn nhất so với tính tay: {lech_sai_phan:.1e}")
assert lech_sai_phan < 1e-8

Ba cách khớp nhau tới sai số làm tròn. Sai phân hữu hạn cần **hai** lượt truyền xuôi cho mỗi
trọng số, nên với $n$ trọng số nó tốn $2n$ lượt. Lan truyền ngược lấy mọi gradient chỉ với một
lượt xuôi và một lượt ngược, bất kể $n$. Với 109 nghìn trọng số ở phần sau, đó là khác biệt
giữa một phép tính và hơn hai trăm nghìn phép tính.

## 4. Cùng cách đếm, cho mạng 784-128-64-10

Mạng nhận dạng chữ số của chương có ba tầng kết nối đầy đủ. Mỗi tầng có `vào × ra` trọng số
cộng `ra` độ lệch (bias), và mỗi mẫu đi qua tốn `vào × ra` phép nhân-cộng (MAC).

In [ ]:
tang = [(784, 128), (128, 64), (64, 10)]
tham_so = [v * r + r for v, r in tang]
mac = [v * r for v, r in tang]

for (v, r), p, m in zip(tang, tham_so, mac):
    print(f"tầng {v:>3} → {r:<3}: {p:>7,} tham số, {m:>7,} MAC mỗi mẫu")

theo_sach("tham số tầng 1", tham_so[0], sach=100_480, nguon="784×128 = 100.352 + 128 = 100.480")
theo_sach("tham số tầng 2", tham_so[1], sach=8_256, nguon="128×64 = 8.192 + 64 = 8.256")
theo_sach("tham số tầng 3", tham_so[2], sach=650, nguon="640 + 10 = 650")
theo_sach("tổng tham số", sum(tham_so), sach=109_386, nguon="109.386 tham số")
theo_sach("tổng MAC mỗi mẫu", sum(mac), sach=109_184, nguon="109.184 MAC")

# Kiểm tra bằng chính PyTorch: một mô hình thật có đúng số tham số đó.
mo_hinh = torch.nn.Sequential(torch.nn.Linear(784, 128), torch.nn.ReLU(),
                              torch.nn.Linear(128, 64), torch.nn.ReLU(),
                              torch.nn.Linear(64, 10))
assert sum(p.numel() for p in mo_hinh.parameters()) == sum(tham_so)

### Bộ nhớ: suy luận so với huấn luyện

Ở FP32 mỗi giá trị chiếm 4 byte. Suy luận chỉ cần trọng số và giá trị kích hoạt của lượt
truyền xuôi. Huấn luyện còn phải giữ gradient (bằng đúng số tham số) và trạng thái của bộ tối
ưu: Adam giữ hai giá trị cho mỗi tham số. Sách đếm một kilobyte là 1,000 byte.

**Dự đoán:** với batch 32, phần nào lớn nhất trong bộ nhớ huấn luyện?

In [ ]:
BYTE = 4
KB = 1_000
batch = 32

kich_hoat = [batch * n for n in (784, 128, 64, 10)]   # đầu vào và đầu ra của mỗi tầng
trong_so_kb = sum(tham_so) * BYTE / KB
kich_hoat_kb = sum(kich_hoat) * BYTE / KB
adam_kb = 2 * sum(tham_so) * BYTE / KB

theo_sach("trọng số FP32", trong_so_kb, sach=437.5, nguon="437,5 KB")
theo_sach("giá trị kích hoạt đầu vào, batch 32", kich_hoat[0] * BYTE / KB, sach=100.4, nguon="100,4 KB")
theo_sach("tổng giá trị kích hoạt lưu lại", sum(kich_hoat), sach=31_552, nguon="31.552 giá trị")
theo_sach("trạng thái Adam", adam_kb, sach=875.1, nguon="875,1 KB")

phan = {"trọng số": trong_so_kb, "giá trị kích hoạt": kich_hoat_kb,
        "gradient": trong_so_kb, "trạng thái Adam": adam_kb}
tong_mb = sum(phan.values()) / 1_000
theo_sach("tổng bộ nhớ huấn luyện (MB)", tong_mb, sach=1.9, nguon="~1,9 MB", tol=0.05)

fig, ax = plt.subplots(figsize=(7, 3.4))
trai = 0.0
for ten, kb in phan.items():
    ax.barh(["huấn luyện"], [kb], left=trai, label=f"{ten}: {kb:,.1f} KB")
    trai += kb
ax.barh(["suy luận"], [trong_so_kb], color="C0")
ax.barh(["suy luận"], [kich_hoat_kb], left=trong_so_kb, color="C1")
ax.set_xlabel("KB (FP32, batch 32)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=2, fontsize=8, frameon=False)
ax.set_title("Bộ nhớ của mạng 784-128-64-10")
plt.tight_layout()
plt.show()

Trạng thái Adam một mình đã gấp đôi trọng số, và gradient thêm một phần bằng trọng số nữa.
Vì thế cùng một mạng cần nhiều bộ nhớ hơn hẳn khi huấn luyện so với khi suy luận, trước cả
khi tính tới giá trị kích hoạt.

### Mật độ tính toán theo batch

Mỗi MAC là hai phép tính (một nhân, một cộng). Đọc trọng số một lần rồi dùng cho cả batch thì
số phép tính trên mỗi byte đọc tăng theo batch. Theo cách đếm lưu lượng của chương (đọc trọng
số một lần, cộng giá trị kích hoạt của cả batch):

In [ ]:
def mat_do(b):
    phep_tinh = 2 * sum(mac) * b
    byte = sum(tham_so) * BYTE + b * sum((784, 128, 64, 10)) * BYTE
    return phep_tinh / byte


theo_sach("mật độ tính toán ở batch 32", mat_do(32), sach=12.4, nguon="12,4 FLOP/byte")
theo_sach("mật độ tính toán ở batch 1", mat_do(1), sach=0.5, nguon="0,5 FLOP/byte")

cac_batch = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
fig, ax = plt.subplots()
ax.semilogx(cac_batch, [mat_do(b) for b in cac_batch], "o-", base=2)
for b, lech in ((1, (8, 8)), (32, (-105, 10))):
    ax.annotate(f"batch {b}: {mat_do(b):.1f}", (b, mat_do(b)), xytext=lech, textcoords="offset points",
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xticks(cac_batch, [str(b) for b in cac_batch])
ax.set_xlabel("kích thước batch")
ax.set_ylabel("FLOP mỗi byte")
ax.set_title("Batch lớn hơn thì mỗi byte trọng số được dùng nhiều lần hơn")
plt.show()

## Thử thêm

1. Đổi `x` ở ô đầu thành `[1.0, 2.0]` rồi chạy lại từ đầu. Nơ-ron ẩn thứ hai có còn bị chặn
   không? Cột thứ hai của gradient bây giờ ra sao? (Các dấu ✗ khi đó là điều được chờ đợi:
   sách tính với đầu vào cũ.)
2. Thay ReLU bằng sigmoid ở cả ba cách tính. Đạo hàm của sigmoid không bao giờ bằng 0, vậy
   trọng số nào bây giờ cũng được cập nhật?
3. Ở phần 4, tăng gấp đôi hai tầng ẩn (784-256-128-10). Số MAC tăng bao nhiêu lần, và vì sao
   không phải bốn lần?

## Tóm lại

* Lan truyền ngược chỉ là quy tắc dây chuyền, áp từ mất mát ngược về từng trọng số, dùng lại
  các giá trị lượt truyền xuôi đã lưu.
* ReLU chặn tín hiệu ở cả hai chiều: nơ-ron tắt ở lượt xuôi thì không nhận gradient ở lượt ngược.
* Autograd và sai phân hữu hạn là hai cách độc lập để kiểm một gradient tính tay.
* Huấn luyện tốn bộ nhớ hơn suy luận vì phải giữ gradient, trạng thái bộ tối ưu và giá trị
  kích hoạt; batch lớn hơn thì mỗi byte trọng số được dùng nhiều lần hơn.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tính toán nơ-ron](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch05-nn-computation/), dựng từ chương
[*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.